<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
LLM-Routing & Provider-Failover
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    install_packages,
    mprint,
    mermaid,
)

setup_api_keys([
    'OPENAI_API_KEY',
    'GOOGLE_API_KEY',
    'GROQ_API_KEY',
], create_globals=False)

print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🔧 Installationen { display-mode: "form" }


install_packages([
    ("langchain-google-genai", "langchain_google_genai"),
    ("langchain-groq", "langchain_groq"),
])

# 1 | Anwendungssituation
---

Ein LLM-Aufruf wirkt oft wie eine einfache Funktion: Prompt hinein, Antwort heraus. In produktiven Anwendungen hängt daran aber ein konkretes Modell bei einem konkreten Anbieter. Wenn dieses Modell nicht erreichbar ist, scheitert der gesamte Aufruf.



<p><font color='darkblue' size="4">
ℹ️ <b>Wichtig</b>
</font></p>

Robustes LLM-Routing trennt deshalb die Anwendung vom einzelnen Modell. Die Anwendung ruft eine Router-Funktion auf. Der Router probiert eine geordnete Modellliste über mehrere Anbieter hinweg und gibt zurück, welches Modell tatsächlich geantwortet hat.



| Risiko | Wirkung | Router-Reaktion |
|---|---|---|
| Modell nicht verfügbar | `403`, `404`, gesperrter Zugriff | Modell überspringen, nächstes Modell nutzen |
| Anbieter überlastet | Timeout, `429`, Serverfehler | kurz warten, einmal erneut versuchen |
| Anfrage fehlerhaft | ungültiger Parameter, zu großer Kontext | Fehler sichtbar machen |

**Grenze**: Routing verbessert Verfügbarkeit, nicht automatisch Qualität. Ein Fallback-Modell kann anders antworten als das Primärmodell.

In [ ]:
#@markdown   <p><font size="4" color='green'>  LLM-Router-Workflow</font> </br></p>

diagram = """
%%{init: {'theme': 'base', 'themeVariables': { 'fontFamily': 'sans-serif', 'fontSize': '13px' }}}%%
flowchart TD
    %% Knoten
    A([Prompt])
    B[LLM-Router.]

    subgraph Models [Modelle & Fallbacks....]
        C[Modell 1]
        I[Modell 2]
        J[Modell 3]
    end

    E{Fehlerklasse...}

    F[Modell 1 sperren....]
    G[Retry..]
    H[Fehler / Exit....]

    D([Result: answer + served_by......])

    %% Fluss
    A --> B
    B --> C

    %% Erfolge
    C -->|OK| D
    I -->|OK| D
    J -->|OK| D

    %% Fehler-Pfade
    C -->|Err..| E
    I -->|Err..| J
    J -->|Err..| H

    E -->|DEAD| F
    E -->|TRANSIENT| G
    E -->|OURS| H

    F --> I
    G -->|Nochmals| C

    %% Styling
    classDef startEnd fill:#2d3748,stroke:#1a202c,stroke-width:2px,color:#fff;
    classDef router fill:#d69e2e,stroke:#b7791f,stroke-width:2px,color:#fff;
    classDef model fill:#3182ce,stroke:#2b6cb0,stroke-width:2px,color:#fff;
    classDef decision fill:#dd6b20,stroke:#c05621,stroke-width:2px,color:#fff;
    classDef success fill:#38a169,stroke:#2f855a,stroke-width:2px,color:#fff;
    classDef error fill:#e53e3e,stroke:#c53030,stroke-width:2px,color:#fff;
    classDef action fill:#718096,stroke:#4a5568,stroke-width:2px,color:#fff;

    class A startEnd;
    class D success;
    class B router;
    class C,I,J model;
    class E decision;
    class F,H error;
    class G action;
"""
mermaid(diagram, width=900)

# 2 | Anti-Pattern: Ein einzelnes Modell
---

Ein fest verdrahtetes Modell ist für erste Experimente in Ordnung. Für robuste Anwendungen ist es fragil: Es gibt keinen zweiten Pfad, wenn genau dieses Modell nicht antwortet.

Das Problem ist nicht der fehlende `try/except`-Block. Das Problem ist die fehlende Alternative.

<p><font color='black' size="5">
❌ Anti-Pattern
</font></p>

In [ ]:
PRIMARY_MODEL = "openai:gpt-5.6-luna"

mprint(f"Das Anti-Pattern verwendet nur `{PRIMARY_MODEL}`.")
mprint("Wenn dieses Modell ausfällt, gibt es keinen alternativen Pfad.")

# 3 | Minimaler LLM-Router
---

Der minimale Router braucht nur vier Bausteine:

1. eine geordnete Modellliste,
2. eine einfache Fehlerklassifikation,
3. einen Circuit Breaker für gerade ausgefallene Modelle,
4. eine Rückgabe mit `answer`, `served_by` und `trace`.

Die Modellstrings folgen derselben Kurznotation wie in M02: `provider:model`. Dadurch kann dieselbe Router-Funktion OpenAI, Google Gemini und Groq ansprechen.

In [ ]:
# ── Stdlib ────────────────────────────────────────────────────────────────────
from dataclasses import dataclass, field
from enum import Enum
import random
import time
from langchain.chat_models import init_chat_model

MODEL_CHAIN = [
    "openai:gpt-5.6-luna",
    "google_genai:gemini-2.5-flash-lite",
    "groq:llama-3.3-70b-versatile",
]

class ModelUnavailableError(Exception):
    """Modell ist nicht erreichbar oder wurde bewusst deaktiviert."""

class ErrorCategory(str, Enum):
    DEAD = "DEAD"
    TRANSIENT = "TRANSIENT"
    OURS = "OURS"

In [ ]:
class CircuitBreaker:
    """Überspringt Modelle für kurze Zeit nach einem DEAD-Fehler."""

    def __init__(self, cooldown_seconds: float = 60.0):
        self.cooldown_seconds = cooldown_seconds
        self.open_until: dict[str, float] = {}

    def is_open(self, model: str) -> bool:
        return time.time() < self.open_until.get(model, 0)

    def open(self, model: str) -> None:
        self.open_until[model] = time.time() + self.cooldown_seconds


def get_status_code(exc: Exception) -> int | None:
    """Liest den HTTP-Statuscode aus einer Provider-Exception, falls vorhanden."""
    status_code = getattr(exc, "status_code", None)
    try:
        return int(status_code) if status_code is not None else None
    except (TypeError, ValueError):
        return None


def classify_error(exc: Exception) -> ErrorCategory:
    """Ordnet typische API-Fehler einer Router-Reaktion zu."""
    status_code = get_status_code(exc)
    message = str(exc).lower()

    if isinstance(exc, ModelUnavailableError) or status_code in {401, 403, 404}:
        return ErrorCategory.DEAD
    if isinstance(exc, TimeoutError) or "timeout" in message or "timed out" in message:
        return ErrorCategory.TRANSIENT
    if status_code in {408, 409, 429, 500, 502, 503, 504}:
        return ErrorCategory.TRANSIENT
    if status_code == 400 or "bad request" in message or "invalid" in message:
        return ErrorCategory.OURS
    return ErrorCategory.TRANSIENT

In [ ]:
MAX_ATTEMPTS = 2

@dataclass
class TraceEvent:
    """Kompakter Trace-Eintrag für Debugging und Auswertung."""
    model: str
    attempt: int | None
    status: str
    error_message: str | None = None


@dataclass
class RouterResult:
    """Ergebnis eines Router-Aufrufs: Antwort, bedienendes Modell und Trace."""
    answer: str
    served_by: str
    trace: list[TraceEvent] = field(default_factory=list)


def call_llm(model: str, prompt: str, forced_dead_models: set[str] | None = None) -> str:
    """Ruft ein echtes Chat-Modell über LangChain auf."""
    forced_dead_models = forced_dead_models or set()
    if model in forced_dead_models:
        raise ModelUnavailableError(f"{model}: im Staging als ausgefallen markiert")

    llm = init_chat_model(model)
    return llm.invoke(prompt).content


def zeige_trace(result: RouterResult) -> None:
    """Gibt den Router-Trace lesbar im Notebook aus."""
    mprint("**trace:**")
    for event in result.trace:
        attempt = "-" if event.attempt is None else event.attempt
        error = f" — {event.error_message}" if event.error_message else ""
        mprint(f"- `{event.model}` | Versuch `{attempt}` | `{event.status}`{error}")


def route_llm(
    prompt: str,
    models: list[str],
    breaker: CircuitBreaker,
    forced_dead_models: set[str] | None = None,
) -> RouterResult:
    """Probiert Modelle nacheinander, bis ein LLM erfolgreich antwortet."""
    trace: list[TraceEvent] = []

    for model in models:
        if breaker.is_open(model):
            trace.append(TraceEvent(model, None, "skipped_circuit_open"))
            continue

        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                answer = call_llm(model, prompt, forced_dead_models)
                trace.append(TraceEvent(model, attempt, "ok"))
                return RouterResult(answer=answer, served_by=model, trace=trace)
            except Exception as exc:
                category = classify_error(exc)
                trace.append(TraceEvent(model, attempt, category.value, str(exc)))

                if category == ErrorCategory.DEAD:
                    breaker.open(model)
                    break
                if category == ErrorCategory.OURS:
                    raise
                if attempt < MAX_ATTEMPTS:
                    time.sleep(0.5)
                    continue
                break

    raise RuntimeError(f"Alle Modelle fehlgeschlagen: {trace}")

# 4 | Echte Modellaufrufe mit kontrolliertem Ausfall
---

Die nächste Zelle ruft echte Modelle auf. Für die Demonstration wird ein zufällig gewähltes Modell aus `MODEL_CHAIN` kontrolliert als ausgefallen markiert. So lässt sich Failover zu den übrigen Anbietern testen, ohne auf einen echten Provider-Ausfall zu warten — und jeder Durchlauf zeigt ein anderes Modell als Ausfall.

`served_by` zeigt, welches Modell tatsächlich geantwortet hat. Der Trace zeigt, welche Modelle versucht oder übersprungen wurden.

In [ ]:
dead_model = random.choice(MODEL_CHAIN)  # Staging-Test: zufällig gewähltes Modell absichtlich abschalten
fallback_models = [m for m in MODEL_CHAIN if m != dead_model]
random.shuffle(fallback_models)  # sonst gewinnt immer das erste MODEL_CHAIN-Modell den Fallback
demo_chain = [dead_model] + fallback_models
FORCED_DEAD_MODELS = {dead_model}

breaker = CircuitBreaker(cooldown_seconds=60)
result = route_llm(
    "Erkläre in zwei Sätzen, warum LLM-Routing eine Anwendung robuster macht.",
    demo_chain,
    breaker,
    forced_dead_models=FORCED_DEAD_MODELS,
)

mprint(f"**served_by:** `{result.served_by}`")
mprint(f"**answer:** {result.answer}")
zeige_trace(result)

fallback_genutzt = result.served_by != dead_model
dead_erkannt = any(
    event.model == dead_model and event.status == "DEAD"
    for event in result.trace
)

mprint(f"**Check - Fallback genutzt:** `{fallback_genutzt}`")
mprint(f"**Check - DEAD erkannt:** `{dead_erkannt}`")

# 5 | Circuit Breaker
---

Nach dem ersten `DEAD`-Fehler ist das ausgefallene Modell im Circuit Breaker. Die nächste Anfrage verschwendet keine Zeit mit demselben kaputten Pfad, sondern startet direkt beim nächsten funktionierenden Anbieter.

Das ist der wichtigste Unterschied zu einer einfachen Retry-Schleife: Ein dauerhaft gesperrtes Modell wird nicht bei jeder Anfrage erneut getestet.

In [ ]:
second_result = route_llm(
    "Nenne drei Risiken, die LLM-Routing nicht automatisch löst.",
    demo_chain,
    breaker,
    forced_dead_models=FORCED_DEAD_MODELS,
)

mprint(f"**served_by:** `{second_result.served_by}`")
zeige_trace(second_result)

circuit_skip_erkannt = any(
    event.model == dead_model and event.status == "skipped_circuit_open"
    for event in second_result.trace
)
fallback_genutzt = second_result.served_by != dead_model

mprint(f"**Check - Circuit Breaker überspringt `{dead_model}`:** `{circuit_skip_erkannt}`")
mprint(f"**Check - Fallback genutzt:** `{fallback_genutzt}`")

# 6 | Akzeptanzkriterien
---

Ein LLM-Router ist erst brauchbar, wenn sein Verhalten messbar ist.

| Kriterium | Erwartung |
|---|---|
| Primärmodell ist `DEAD` | Anfrage wird von einem Fallback-Modell beantwortet |
| Wiederholte Anfrage nach `DEAD` | Primärmodell wird per Circuit Breaker übersprungen |
| `TRANSIENT` | Es gibt höchstens einen Retry pro Modell |
| `OURS` | Fehler wird sichtbar gemeldet und nicht durch Fallback verdeckt |
| Trace | Jede Anfrage protokolliert, welches Modell geantwortet hat |

In Staging sollte das Primärmodell regelmäßig absichtlich deaktiviert werden. Ein nie getesteter Fallback ist nur eine Annahme.

In [ ]:
# Selbstcheck: Fehler in der eigenen Anfrage werden nicht kaschiert
class FakeBadRequest(Exception):
    status_code = 400


category = classify_error(FakeBadRequest("invalid request"))
ours_erkannt = category == ErrorCategory.OURS
mprint(f"**Erwartete Kategorie:** `{category.value}`")
mprint(f"**Check - OURS erkannt:** `{ours_erkannt}`")


# Selbstcheck: Timeouts werden als TRANSIENT klassifiziert und damit genau einmal wiederholt
timeout_category = classify_error(TimeoutError("request timed out"))
timeout_erkannt = timeout_category == ErrorCategory.TRANSIENT
mprint(f"**Timeout-Kategorie:** `{timeout_category.value}`")
mprint(f"**Check - Timeout ist TRANSIENT:** `{timeout_erkannt}`")

# A | Aufgabe
---

**Grundlagen**

Ändere die Reihenfolge in `MODEL_CHAIN`, z. B. Google zuerst und OpenAI als Fallback. Prüfe anschließend im Trace, ob die Reihenfolge stimmt.



**Aufbau**

Ergänze eine neue Fehlerklasse für Timeouts. Sie soll als `TRANSIENT` behandelt werden und genau einen Retry auslösen.



**Vertiefung**

Baue eine kleine Auswertung über mehrere Router-Aufrufe: Wie oft antwortet welches Modell? Welche Fehlerklasse tritt am häufigsten auf? Gib am Ende eine Tabelle mit `served_by`, Anzahl und Anteil aus. Für kontrollierte Ausfälle kann `FORCED_DEAD_MODELS` variiert werden.

# B | Dokumente zum Weiterlesen
---

- [Your Model Can Vanish Overnight. Your Agent Shouldn’t.](https://medium.com/@Micheal-Lanham/your-model-can-vanish-overnight-your-agent-shouldnt-6c516e272bb7)
- [LangChain: Model initialization](https://python.langchain.com/docs/how_to/chat_models_universal_init/)
- [LangChain: Fallbacks](https://python.langchain.com/docs/how_to/fallbacks/)